# Tiny Mamba Memory Research: Colab Reproduction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/niladridutta2001/tiny-mamba-memory-research/blob/master/experiment.ipynb)

This notebook is the executable companion to the research report. It generates the assignment data, checks causal/query-only invariants, trains the original Mamba baseline, the key-conditioned write-routing proposal, and its parameter-matched tied-routing ablation, evaluates all three on fixed data, and visualizes the main and supplementary experiments.

**Research integrity.** Newly computed outputs are kept separate from the archived measurements reported in the paper. Long-horizon probes trained with long-horizon labels are diagnostic decodability measurements, not strict short-only generalization results.

## 0. How to run

Use a Colab GPU and select **Runtime > Run all**. `assignment` uses the requested 300-update starting budget. `report` uses the disclosed 1,500-update extension that produced the paper tables. `smoke` is only a short installation/CPU demonstration and must not be reported as the mandatory baseline.

The repository is currently private, so the setup cell asks for a read-only GitHub token using hidden input. No token is written to the notebook, remote URL, or output. Once the repository is public, leave the prompt blank.

In [ ]:
# Acquire the repository in a fresh Colab runtime without persisting credentials.
from pathlib import Path
import getpass, io, os, shutil, urllib.request, zipfile

OWNER = "niladridutta2001"
REPOSITORY = "tiny-mamba-memory-research"
COLAB_ROOT = Path("/content")
local_candidate = Path.cwd()
target = COLAB_ROOT / REPOSITORY

if (local_candidate / "model.py").exists():
    repo = local_candidate
elif (target / "model.py").exists():
    repo = target
else:
    token = getpass.getpass("GitHub read token (hidden; blank if repository is public): ")
    request = urllib.request.Request(
        f"https://api.github.com/repos/{OWNER}/{REPOSITORY}/zipball/master",
        headers={"Accept": "application/vnd.github+json", **({"Authorization": f"Bearer {token}"} if token else {})},
    )
    with urllib.request.urlopen(request) as response:
        archive = zipfile.ZipFile(io.BytesIO(response.read()))
    extracted_name = archive.namelist()[0].split("/")[0]
    archive.extractall(COLAB_ROOT)
    extracted = COLAB_ROOT / extracted_name
    if target.exists():
        shutil.rmtree(target)
    extracted.rename(target)
    repo = target
    del token

os.chdir(repo)
print(f"Repository: {repo}")

In [ ]:
%pip install -q einops pyyaml pandas matplotlib pytest

# Colab already supplies an accelerator-compatible PyTorch build. The project
# modules are imported directly, avoiding a needless torch downgrade.

In [ ]:
import json, random, subprocess, sys, time
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

RUN_MODE = "assignment"  # one of: smoke, assignment, report
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SETTINGS = {
    "smoke": {"preset": "small", "updates": 5, "examples": 64, "bootstrap": 200, "eval_batch": 4},
    "assignment": {"preset": "assignment", "updates": 300, "examples": 256, "bootstrap": 2000, "eval_batch": 4},
    "report": {"preset": "assignment", "updates": 1500, "examples": 1024, "bootstrap": 10000, "eval_batch": 4},
}[RUN_MODE]
RUN_ROOT = Path("results/colab") / RUN_MODE / f"seed{SEED}"
DATA_DIR = Path("data/generated")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print({"mode": RUN_MODE, "device": DEVICE, "torch": torch.__version__, **SETTINGS})
if DEVICE == "cpu":
    print("CPU path is explicitly labeled. Use smoke mode for a short demonstration; full comparisons preserve the mandatory 32/8 architecture.")

## 1. Architecture and causal invariants

Each event ID is embedded, projected to the 64-dimensional inner width, passed through a causal depthwise convolution and SiLU, and then enters the selective scan. At timestep $t$, $\exp(\Delta_t A)$ controls recurrent retention, $B_t$ controls the input write, and $C_t$ contracts the eight state modes into the output path. The residual block returns a 32D feature and `Linear(32,5)` predicts UNSET or values 1--4.

The implementation must preserve left-to-right processing, causal convolution, the 41 logical tokens (padded internally to 48 embeddings), and loss only at QUERY positions. The eight logical keys share recurrent coordinates, so retention, overwrite correctness, cross-key interference, and readout accessibility can fail independently.

## 2. Data generation and leakage checks

The assignment preset creates 4,000 training sequences and 500 examples for every validation/test split. Inputs contain only event tokens; labels are present only at QUERY positions. The exact dictionary oracle is defined during generation and is 100% correct by construction.

In [ ]:
manifest_path = DATA_DIR / "manifest.json"
regenerate = True
if manifest_path.exists():
    existing = json.loads(manifest_path.read_text())
    regenerate = existing.get("preset") != SETTINGS["preset"]
if regenerate:
    subprocess.run([sys.executable, "generate_dataset.py", "--preset", SETTINGS["preset"], "--output-dir", str(DATA_DIR)], check=True)
manifest = json.loads(manifest_path.read_text())
pd.DataFrame(manifest["splits"]).T

In [ ]:
# Required automated checks: oracle/NOOP/overwrite semantics, leakage,
# query masking, shapes, gradients, causality, and changed interfaces.
completed = subprocess.run([sys.executable, "-m", "pytest", "-q"], text=True, capture_output=True)
print(completed.stdout)
if completed.returncode:
    print(completed.stderr)
    raise RuntimeError("Automated checks failed")

In [ ]:
import gzip
with gzip.open(DATA_DIR / "train.jsonl.gz", "rt", encoding="utf-8") as source:
    sample = json.loads(next(source))
query_rows = [(i, event, target) for i, (event, target) in enumerate(zip(sample["events"], sample["query_targets"])) if target is not None]
print("First 12 events:", sample["events"][:12])
print("First five supervised QUERY positions:", query_rows[:5])
assert all((event[0] == "QUERY") == (target is not None) for event, target in zip(sample["events"], sample["query_targets"]))

## 3. Primary controlled experiment

**Hypothesis.** Unrelated-key SETs interfere because multiple keys write through overlapping recurrent channels. The proposed $8\times64$ routing table gates only $\Delta_tB_tu_t$ according to the SET key.

**Decisive ablation.** The tied-routing control retains the identical table, parameter count, training budget, and computation, but averages its rows before use. It removes key selectivity without removing capacity. The predeclared hypothesis is falsified unless key-conditioned routing exceeds this control by at least 2 percentage points under dense interference.

All variants use seed 42, identical data, AdamW, FP32, batch 16, LR $10^{-3}$, and lengths 32--128. `assignment` starts at 300 updates; `report` reproduces the disclosed 1,500-update extension.

In [ ]:
MODEL_CONFIGS = {
    "Baseline": Path("configs/baseline.yaml"),
    "Key-conditioned": Path("configs/proposed.yaml"),
    "Tied routing": Path("configs/ablation.yaml"),
}
CHECKPOINTS = {}
for name, config in MODEL_CONFIGS.items():
    output = RUN_ROOT / "checkpoints" / name.lower().replace(" ", "_").replace("-", "_")
    checkpoint = output / "baseline.pt"
    CHECKPOINTS[name] = checkpoint
    if not checkpoint.exists():
        command = [sys.executable, "train.py", "--config", str(config), "--data-dir", str(DATA_DIR), "--output-dir", str(output), "--device", DEVICE, "--seed", str(SEED), "--max-updates", str(SETTINGS["updates"])]
        if RUN_MODE == "smoke": command += ["--max-validation-batches", "2"]
        print("Training", name)
        subprocess.run(command, check=True)
    else:
        print("Reusing", checkpoint)

In [ ]:
training_rows = []
for name, checkpoint in CHECKPOINTS.items():
    metadata = json.loads((checkpoint.parent / "baseline_validation.json").read_text())
    training_rows.append({
        "model": name, "parameters": metadata["parameter_count"],
        "updates": metadata["training"]["ending_update"],
        "train_seconds": metadata["training"]["elapsed_seconds"],
        "validation_accuracy": metadata["validation"]["query_accuracy"],
        "peak_gpu_MiB": metadata["peak_gpu_memory_bytes"] / 2**20,
        "hardware": metadata["device_name"],
    })
training_table = pd.DataFrame(training_rows)
training_table.style.format({"validation_accuracy": "{:.2%}", "train_seconds": "{:.1f}", "peak_gpu_MiB": "{:.1f}"})

## 4. Fixed-split evaluation

Every model is evaluated on the same ID-128, OOD-256, OOD-512, delay-stress-512, and overwrite-stress-512 files. The baseline evaluation also records Always-UNSET, training-majority, and exact-oracle controls. Aggregate query accuracy is not the same as a 511-step dependency: ordinary length-512 sequences contain many recent assignments.

In [ ]:
evaluation_frames = []
for name, checkpoint in CHECKPOINTS.items():
    output = RUN_ROOT / "fixed_evaluation" / name.lower().replace(" ", "_").replace("-", "_")
    subprocess.run([sys.executable, "evaluate.py", "--checkpoint", str(checkpoint), "--data-dir", str(DATA_DIR), "--output-dir", str(output), "--batch-size", str(SETTINGS["eval_batch"]), "--device", DEVICE], check=True)
    frame = pd.read_csv(output / "a3_metrics.csv")
    frame["model"] = name
    evaluation_frames.append(frame)
fixed = pd.concat(evaluation_frames, ignore_index=True)
neural = fixed[fixed.baseline == "Neural"].pivot(index="model", columns="split", values="query_accuracy")
controls = fixed[fixed.model == "Baseline"].pivot(index="baseline", columns="split", values="query_accuracy")
display(controls.style.format("{:.2%}").set_caption("Baseline, constants, and exact oracle"))
display(neural.style.format("{:.2%}").set_caption("Matched neural variants"))

In [ ]:
# Reuse the repository's paired H1/H2 design, but point it at the checkpoints
# trained above. This produces retention-by-delay, interference, routing
# geometry, fixed-split, and paired-bootstrap records.
import part_b_compare
part_b_compare.MODEL_PATHS = CHECKPOINTS
comparison_dir = RUN_ROOT / "primary_comparison"
comparison = part_b_compare.run(SimpleNamespace(
    output_dir=comparison_dir, data_dir=DATA_DIR, examples=SETTINGS["examples"],
    batch_size=64 if DEVICE == "cuda" else 8, fixed_batch_size=SETTINGS["eval_batch"],
    bootstrap_draws=SETTINGS["bootstrap"], evaluation_seed=20261023,
    bootstrap_seed=424242, device=DEVICE,
))
display(pd.DataFrame(comparison["fixed_splits"]).pivot(index="model", columns="split", values="query_accuracy").style.format("{:.2%}"))
dense_effect = pd.DataFrame(comparison["contrasts"]).query("density == 1.0 and comparison == 'proposed_vs_tied'")
display(dense_effect[["delay", "effect_pp", "ci_low", "ci_high"]].style.format({c: "{:+.2f}" for c in ["effect_pp", "ci_low", "ci_high"]}).set_caption("Key-conditioned minus tied routing at 100% SET density (percentage points)"))

In [ ]:
from IPython.display import Image, display
for filename in ["part_b_h2_accuracy.png", "part_b_h2_effects.png", "routing_heatmap.png"]:
    path = comparison_dir / filename
    if path.exists(): display(Image(filename=str(path)))

## 5. Overwrite count and stale-value errors

The final probe in each overwrite-stress sequence asks for the newest value after 1, 2, 4, or 8 overwrites. A stale-value error is specifically a prediction equal to a previously valid but now obsolete value; consecutive values are forced to differ.

In [ ]:
from data.dataset import StateTrackingDataset
from evaluate import load_model

overwrite_data = StateTrackingDataset(DATA_DIR / "stress_overwrite_512.jsonl.gz")
overwrite_rows = []
for model_name, checkpoint in CHECKPOINTS.items():
    model = load_model(checkpoint, torch.device(DEVICE)).eval()
    for count in (1, 2, 4, 8):
        records = [record for record in overwrite_data.records if record["overwrite_count"] == count]
        correct = stale = total = 0
        for start in range(0, len(records), SETTINGS["eval_batch"]):
            batch = records[start:start + SETTINGS["eval_batch"]]
            tokens = torch.tensor([[__import__("generate_dataset").encode_event(e) for e in item["events"]] for item in batch], device=DEVICE)
            with torch.no_grad(): prediction = model(tokens)[:, -1].argmax(-1).cpu().tolist()
            for item, pred in zip(batch, prediction):
                writes = [event[2] for event in item["events"] if event[0] == "SET" and event[1] == item["events"][-1][1]]
                newest, obsolete = writes[-1], set(writes[:-1])
                correct += pred == newest; stale += pred in obsolete; total += 1
        overwrite_rows.append({"model": model_name, "overwrite_count": count, "accuracy": correct / total, "stale_error": stale / total, "n": total})
overwrite_table = pd.DataFrame(overwrite_rows)
display(overwrite_table.pivot(index="model", columns="overwrite_count", values="accuracy").style.format("{:.1%}"))
display(overwrite_table.groupby("model")[["accuracy", "stale_error"]].mean().style.format("{:.1%}"))

## 6. Reported mechanistic experiments

The following panels are the immutable, tracked artifacts used in the report. They cover the longer multi-seed and internal-state studies that are impractical to retrain during every notebook review. Their source scripts, fixed seeds, CSV/JSON records, and tests are in this repository. Displaying these artifacts does not claim they were recomputed in this runtime.

In [ ]:
from PIL import Image as PILImage

def show_panel(paths, titles, columns=2, width=15):
    rows = int(np.ceil(len(paths) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(width, 5.2 * rows), squeeze=False)
    for axis, path, title in zip(axes.flat, paths, titles):
        axis.imshow(PILImage.open(path)); axis.set_title(title); axis.axis("off")
    for axis in axes.flat[len(paths):]: axis.axis("off")
    plt.tight_layout(); plt.show()

### H1/H2/H4/H5: from behavioral failure to readout localization

H1 balances keys and values and inserts only NOOPs: accuracy falls to chance at delay 500. H2 replaces NOOPs with unrelated SET or QUERY events at matched positions: SETs are worse than QUERYs before the task reaches its floor. H4 then shows that forcing identity immediately is harmful because it truncates convolutional settling, whereas identity after the three-step width-4 washout preserves about 97% retrieval. H5 finds that reducing attenuation can leave information decodable from long-horizon features even when the original short-trained head remains near chance, establishing a retention/readout mismatch.

In [ ]:
figure_dir = Path("overleaf_report/figures")
show_panel(
    [figure_dir / "retention_by_delay.png", figure_dir / "cross_key_interference.png", figure_dir / "noop_identity_diagnostic.png", figure_dir / "readout_decodability.png"],
    ["H1: balanced passive retention", "H2: cross-key interference", "H4: convolution-aware identity intervention", "H5: internal readout decodability"],
)

### Supplementary adaptive-kernel diagnostic

A shared width-8 depthwise kernel is softly masked to effective widths 1, 2, 4, or 8. The static mixture learns one global mixture; the adaptive mixture conditions it on the current event. Both have 10,729 parameters. Adaptive routing improves aggregate assignment stress accuracy but does not beat the matched static mixture on the controlled final-gap/final-overwrite probes; extreme retention remains near chance. Its modest memory premium comes with a substantial runtime premium.

In [ ]:
show_panel(
    [figure_dir / "adaptive_assignment_stress.png", figure_dir / "adaptive_kernel_comparison.png", figure_dir / "kernel_resource_tradeoff.png"],
    ["Assignment stress and final probes", "Static versus adaptive diagnostic", "Training time and peak GPU memory"],
)

### H7/H8: trained slow-memory recurrence

H7 partitions four fast and four slow SSM modes and constrains their effective rates. H8 separates slow-state erasure $r_t$ from writing $w_t$ and compares it with an exactly parameter-matched tied gate $r_t=w_t$. The independent-gate prediction is falsified in the seed-42 pilot: it erases more on NOOP than SET and underperforms the tied control at delay 511. The tied gate learns the cleaner event-selective policy and combines 100% post-32 overwrite accuracy with 0% stale errors in the controlled probe. These architecture results require multi-seed confirmation.

In [ ]:
show_panel(
    [figure_dir / "h8_retention_accuracy.png", figure_dir / "h8_event_gates.png", figure_dir / "h8_overwrite_tradeoff.png", figure_dir / "h8_drift_parameters.png"],
    ["Original-head retention", "Event-conditioned gate audit", "Retention/overwrite/SVER trade-off", "Drift and parameter control"],
)

## 7. Full diagnostic reproduction map

The primary three-model experiment above is runnable top-to-bottom. To regenerate every report artifact, use `report` mode for the primary checkpoints and then run the repository scripts documented in `README.md`: `h2_multiseed.py`, `interference_term_ablation.py`, `h4_noop_identity.py`, `decay_only_retention_sweep.py`, `h5b_internal_readout.py`, `adaptive_kernel_compare.py`, `assignment_stress_evaluation.py`, `h7a_pilot.py`, `h7b_identity_gate.py`, and `h8_decoupled_retention.py`. These experiments require their independently trained seed/checkpoint grids; the tracked CSV/JSON files disclose the exact evaluation seeds, counts, and bootstrap settings.

In [ ]:
reproduction_record = {
    "run_mode": RUN_MODE, "seed": SEED, "device": DEVICE,
    "hardware": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "torch": torch.__version__, "settings": SETTINGS,
    "checkpoints": {name: str(path) for name, path in CHECKPOINTS.items()},
    "note": "smoke mode is not a reportable mandatory-baseline result" if RUN_MODE == "smoke" else "mandatory dimensions preserved",
}
(RUN_ROOT / "reproduction_record.json").write_text(json.dumps(reproduction_record, indent=2))
reproduction_record

## 8. Findings and final reflection

1. Aggregate OOD-512 accuracy overstates true long-horizon memory because it is dominated by queries with recent SETs. Balanced final-query accuracy reaches chance after a 500-step passive gap.
2. Unrelated SETs cause more damage than matched QUERY activity at usable delays, but suppressing incoming writes and key-specific routing do not solve the problem. The proposed key router is matched by its tied control and learns nearly identical rows, so its mechanism is falsified.
3. The strongest causal result is the convolution-aware H4 intervention: after allowing three timesteps for the width-4 convolution to settle, preventing further passive recurrent evolution rescues retrieval.
4. H5 shows a secondary readout problem: changed retention can preserve decodable information while the original short-trained head remains miscalibrated.
5. The seed-42 tied slow-memory gate is the most promising trained mechanism, but it is not yet a multi-seed conclusion.

**Best-supported bottleneck:** harmful passive recurrent evolution/retention is primary, with horizon-dependent readout alignment secondary. **Next experiment:** repeat the tied slow-memory gate over multiple training seeds against the constrained fast/slow model and independent-gate control, using the same delay, overwrite, SVER, runtime, memory, and probe protocol.

**Attribution and AI disclosure:** the Mamba backbone derives from John Ma's `mamba-minimal` repository under its retained license. The assignment generator was supplied with the task. AI assistance was used for implementation, experiment orchestration, visualization, and report editing; all code, measurements, and claims remain inspectable in this repository.